# Bayesian Inference with PyMC

This notebook demonstrates how to use PyMC to perform Bayesian inference and model posterior distributions.

## Topics Covered:
1. Introduction to PyMC
2. Simple coin flipping example (Beta-Binomial)
3. Estimating a normal mean
4. Visualization of priors and posteriors
5. Posterior summaries and credible intervals

In [ ]:
# !pip install pymc arviz seaborn graphviz

In [ ]:
# Import necessary libraries
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")

## Example 1: Coin Flipping (Beta-Binomial Model)

**Scenario:** We flip a coin 100 times and observe 65 heads. What is the probability that the coin lands heads?

**Bayesian Model:**
- Prior: $\theta \sim \text{Beta}(2, 2)$ (slightly informative, centered at 0.5)
- Likelihood: $X \sim \text{Binomial}(n=100, p=\theta)$
- Posterior: $\theta | X \sim \text{Beta}(2 + 65, 2 + 35)$ (analytically)

We'll use PyMC to sample from the posterior distribution.

In [ ]:
# Observed data
n_flips = 100
n_heads = 65

print(f"Data: {n_heads} heads out of {n_flips} flips")
print(f"Sample proportion: {n_heads/n_flips:.3f}")

In [ ]:
# Define the Bayesian model
with pm.Model() as coin_model:
    # Prior distribution for theta (probability of heads)
    theta = pm.Beta("theta", alpha=2, beta=2)

    # Likelihood (sampling distribution)
    obs = pm.Binomial("obs", n=n_flips, p=theta, observed=n_heads)

    # Sample from the posterior
    trace = pm.sample(2000, tune=1000, return_inferencedata=True, random_seed=42)

print("\nModel summary:")
print(pm.model_to_graphviz(coin_model))

In [ ]:
# Display posterior summary statistics
print(az.summary(trace, var_names=["theta"]))

### Interpretation of Results

The summary shows:
- **mean**: Posterior mean (expected value of $\theta$ given the data)
- **sd**: Posterior standard deviation (uncertainty in our estimate)
- **hdi_3%** and **hdi_97%**: 94% Highest Density Interval (credible interval)
- **ess_bulk/ess_tail**: Effective sample size (should be large)
- **r_hat**: Convergence diagnostic (should be close to 1.0)

A 94% credible interval means: "Given the data, there's a 94% probability that $\theta$ lies in this interval."

In [ ]:
# Plot 1: Trace plot (check for convergence)
az.plot_trace(trace, var_names=["theta"])
plt.tight_layout()
plt.show()

In [ ]:
# Compare prior, likelihood, and posterior
theta_vals = np.linspace(0, 1, 1000)

# Prior: Beta(2, 2)
from scipy.stats import beta

prior = beta(2, 2).pdf(theta_vals)

# Likelihood (normalized for visualization)
likelihood = beta(n_heads + 1, n_flips - n_heads + 1).pdf(theta_vals)

# Analytical posterior: Beta(2 + 65, 2 + 35)
analytical_posterior = beta(2 + n_heads, 2 + n_flips - n_heads).pdf(theta_vals)

# PyMC sampled posterior
sampled_theta = trace.posterior["theta"].values.flatten()

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(theta_vals, prior, label="Prior: Beta(2, 2)", linewidth=2, linestyle="--")
ax.plot(
    theta_vals, likelihood, label="Likelihood (normalized)", linewidth=2, linestyle=":"
)
ax.plot(
    theta_vals,
    analytical_posterior,
    label="Analytical Posterior: Beta(67, 37)",
    linewidth=2,
    color="red",
)
ax.hist(
    sampled_theta,
    bins=50,
    density=True,
    alpha=0.3,
    label="PyMC Sampled Posterior",
    color="green",
)

ax.axvline(
    n_heads / n_flips,
    color="black",
    linestyle="--",
    label=f"MLE: {n_heads/n_flips:.3f}",
    alpha=0.5,
)
ax.set_xlabel("θ (Probability of Heads)", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("Prior, Likelihood, and Posterior for Coin Flip Example", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Posterior predictive check
# What would new data look like given our posterior beliefs?

with coin_model:
    posterior_predictive = pm.sample_posterior_predictive(trace, random_seed=42)

# Extract posterior predictive samples
pp_samples = posterior_predictive.posterior_predictive["obs"].values.flatten()

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(pp_samples, bins=30, density=True, alpha=0.6, edgecolor="black")
ax.axvline(
    n_heads,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Observed: {n_heads} heads",
)
ax.set_xlabel("Number of Heads (out of 100 flips)", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("Posterior Predictive Distribution", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Posterior predictive mean: {pp_samples.mean():.2f}")
print(f"Posterior predictive std: {pp_samples.std():.2f}")

## Example 2: Estimating a Normal Mean

**Scenario:** We measure the heights of 20 students (in cm). What is the average height?

**Bayesian Model:**
- Data: $X_1, \ldots, X_{20} \sim N(\mu, \sigma^2)$
- Prior on mean: $\mu \sim N(170, 10^2)$ (weak prior centered at 170 cm)
- Prior on std: $\sigma \sim \text{HalfNormal}(10)$ (must be positive)
- Posterior: $(\mu, \sigma) | X$ (computed via MCMC)

In [ ]:
# Generate synthetic data (in reality, this would be observed)
true_mean = 175
true_std = 8
n_students = 20

heights = np.random.normal(true_mean, true_std, n_students)

print(f"Observed heights (cm): {heights}")
print(f"\nSample mean: {heights.mean():.2f}")
print(f"Sample std: {heights.std():.2f}")

In [ ]:
# Define the Bayesian model
with pm.Model() as height_model:
    # Priors
    mu = pm.Normal("mu", mu=170, sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=10)

    # Likelihood
    obs = pm.Normal("obs", mu=mu, sigma=sigma, observed=heights)

    # Sample from posterior
    trace_height = pm.sample(2000, tune=1000, return_inferencedata=True, random_seed=42)

In [ ]:
# Posterior summary
print(az.summary(trace_height, var_names=["mu", "sigma"]))

In [ ]:
# Visualize posterior distributions
az.plot_posterior(
    trace_height, var_names=["mu", "sigma"], hdi_prob=0.95, figsize=(12, 4)
)
plt.tight_layout()
plt.show()

In [ ]:
# Joint posterior distribution
fig, ax = plt.subplots(figsize=(8, 8))

mu_samples = trace_height.posterior["mu"].values.flatten()
sigma_samples = trace_height.posterior["sigma"].values.flatten()

# Create hexbin plot
hb = ax.hexbin(mu_samples, sigma_samples, gridsize=50, cmap="Blues", mincnt=1)
ax.set_xlabel("μ (Mean Height)", fontsize=12)
ax.set_ylabel("σ (Std Deviation)", fontsize=12)
ax.set_title("Joint Posterior Distribution of μ and σ", fontsize=14)
ax.axvline(true_mean, color="red", linestyle="--", label=f"True μ = {true_mean}")
ax.axhline(true_std, color="red", linestyle="--", label=f"True σ = {true_std}")
ax.legend()
plt.colorbar(hb, ax=ax, label="Density")
plt.tight_layout()
plt.show()

## Example 3: Linear Regression with Uncertainty

**Scenario:** Predict exam scores based on study hours.

**Bayesian Model:**
- $y_i = \alpha + \beta \cdot x_i + \epsilon_i$
- $\epsilon_i \sim N(0, \sigma^2)$
- Priors: $\alpha \sim N(0, 20)$, $\beta \sim N(0, 20)$, $\sigma \sim \text{HalfNormal}(10)$

This gives us uncertainty estimates for the slope and intercept!

In [ ]:
# Generate synthetic data
np.random.seed(123)
n_students_reg = 50
study_hours = np.random.uniform(0, 10, n_students_reg)

# True relationship: score = 50 + 4 * hours + noise
true_alpha = 50
true_beta = 4
true_sigma_reg = 8

exam_scores = (
    true_alpha
    + true_beta * study_hours
    + np.random.normal(0, true_sigma_reg, n_students_reg)
)

# Visualize data
plt.figure(figsize=(10, 6))
plt.scatter(study_hours, exam_scores, alpha=0.6, s=50)
plt.xlabel("Study Hours", fontsize=12)
plt.ylabel("Exam Score", fontsize=12)
plt.title("Exam Scores vs Study Hours", fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Define Bayesian linear regression model
with pm.Model() as regression_model:
    # Data as mutable (so we can update for predictions)
    study_hours_data = pm.Data("study_hours_data", study_hours)

    # Priors
    alpha = pm.Normal("alpha", mu=0, sigma=20)
    beta = pm.Normal("beta", mu=0, sigma=20)
    sigma = pm.HalfNormal("sigma", sigma=10)

    # Expected value
    mu_reg = alpha + beta * study_hours_data

    # Likelihood
    scores = pm.Normal("scores", mu=mu_reg, sigma=sigma, observed=exam_scores)

    # Sample
    trace_reg = pm.sample(2000, tune=1000, return_inferencedata=True, random_seed=42)

In [ ]:
# Summary of regression parameters
print(az.summary(trace_reg, var_names=["alpha", "beta", "sigma"]))

In [ ]:
# Visualize posterior distributions of parameters
az.plot_posterior(
    trace_reg, var_names=["alpha", "beta", "sigma"], hdi_prob=0.95, figsize=(14, 4)
)
plt.tight_layout()
plt.show()

In [ ]:
# Plot regression lines with uncertainty
fig, ax = plt.subplots(figsize=(12, 7))

# Data points
ax.scatter(study_hours, exam_scores, alpha=0.6, s=50, label="Observed Data", zorder=3)

# Get posterior samples
alpha_samples = trace_reg.posterior["alpha"].values.flatten()
beta_samples = trace_reg.posterior["beta"].values.flatten()

# Plot many regression lines from posterior (shows uncertainty)
x_range = np.linspace(0, 10, 100)
for i in np.random.choice(len(alpha_samples), 100):
    y_line = alpha_samples[i] + beta_samples[i] * x_range
    ax.plot(x_range, y_line, "gray", alpha=0.02)

# Plot mean regression line
mean_alpha = alpha_samples.mean()
mean_beta = beta_samples.mean()
y_mean = mean_alpha + mean_beta * x_range
ax.plot(
    x_range,
    y_mean,
    "red",
    linewidth=3,
    label=f"Mean: y = {mean_alpha:.1f} + {mean_beta:.2f}x",
)

# True regression line
y_true = true_alpha + true_beta * x_range
ax.plot(
    x_range,
    y_true,
    "blue",
    linewidth=2,
    linestyle="--",
    label=f"True: y = {true_alpha} + {true_beta}x",
)

ax.set_xlabel("Study Hours", fontsize=12)
ax.set_ylabel("Exam Score", fontsize=12)
ax.set_title("Bayesian Linear Regression with Uncertainty", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Posterior predictive: predict for a new student who studies 7 hours
hours_new = 7

with regression_model:
    # Update the study hours to the new value
    pm.set_data({"study_hours_data": np.array([hours_new])})

    # Sample posterior predictive for the new data point
    trace_pred = pm.sample_posterior_predictive(trace_reg, random_seed=42)

predicted_scores = trace_pred.posterior_predictive["scores"].values.flatten()

# Visualize prediction
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(predicted_scores, bins=40, density=True, alpha=0.7, edgecolor="black")
ax.axvline(
    predicted_scores.mean(),
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Mean prediction: {predicted_scores.mean():.1f}",
)
ax.axvline(
    true_alpha + true_beta * hours_new,
    color="blue",
    linestyle="--",
    linewidth=2,
    label=f"True value: {true_alpha + true_beta * hours_new:.1f}",
)
ax.set_xlabel("Predicted Exam Score", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title(
    f"Posterior Predictive Distribution for {hours_new} Study Hours", fontsize=14
)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 95% credible interval
lower, upper = np.percentile(predicted_scores, [2.5, 97.5])
print(f"\nPrediction for student studying {hours_new} hours:")
print(f"Mean: {predicted_scores.mean():.2f}")
print(f"95% Credible Interval: [{lower:.2f}, {upper:.2f}]")